# ViSceT5 — Finetune từ **checkpoint tốt nhất** của Pretrain
Chạy tuần tự. Chỉ cần điền **HF token** ở cell cấu hình.

In [ ]:
!git clone https://github.com/Kussssssss/ViSceT5.git
%cd ViSceT5
!git pull

In [ ]:
!bash setup.sh
# Nếu Colab báo cần restart: Runtime > Restart, rồi chạy tiếp TỪ cell cấu hình.

In [ ]:
import os
os.environ['HF_TOKEN'] = 'hf_xxx'          # <== ĐIỀN token HF của bạn
HF_PRETRAIN_REPO = 'Kus669/ViSceT5-pretrain-1epoch'   # repo chứa MODEL ĐÃ PRETRAIN
HF_FINETUNE_REPO = 'Kus669/ViSceT5-finetune'          # repo sẽ lưu model finetune

In [ ]:
import argparse
from scripts import prepare_dataset
prepare_dataset.main(argparse.Namespace(config='configs/data/ViTextVQA.yaml', data_dir='./datasets'))

In [ ]:
from scripts import init_model
init_model.main()

### Xác định & tải CHECKPOINT TỐT NHẤT
Best checkpoint lấy từ `trainer_state.json` (trường `best_model_checkpoint`, do HF tính theo `metric_for_best_model`). In rõ checkpoint nào + metric để chắc chắn finetune đúng bản.

In [ ]:
import os, re, json
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

api = HfApi(token=os.environ['HF_TOKEN'])
files = api.list_repo_files(HF_PRETRAIN_REPO, repo_type='model')
ckpts = sorted({int(m.group(1)) for f in files for m in [re.match(r'checkpoint-(\d+)/', f)] if m})
print('Checkpoints tren repo:', ckpts)

best_name, best_metric, best_step = None, None, None
if ckpts:
    ts = hf_hub_download(HF_PRETRAIN_REPO, f'checkpoint-{ckpts[-1]}/trainer_state.json',
                         repo_type='model', token=os.environ['HF_TOKEN'])
    st = json.load(open(ts))
    best_metric = st.get('best_metric')
    bmc = st.get('best_model_checkpoint')            # vd: .../checkpoint-6300
    print(f'Train toi epoch={round(st.get("epoch",0),3)} step={st.get("global_step")} | '
          f'best_metric={best_metric} | best_model_checkpoint={bmc}')
    if bmc:
        cand = os.path.basename(str(bmc).rstrip('/'))
        if f'{cand}/model.safetensors' in files:
            best_name = cand
            best_step = int(re.match(r'checkpoint-(\d+)', cand).group(1))

# Ưu tiên đúng best checkpoint; nếu bị prune khỏi repo thì dùng model ở gốc
# (root = save_model sau load_best_model_at_end = cùng là best); cuối cùng mới lấy step lớn nhất.
if best_name:
    subdir = best_name
    print(f'==> DÙNG BEST checkpoint: {subdir} (metric={best_metric})')
elif 'model.safetensors' in files:
    subdir = ''
    print(f'==> best_model_checkpoint da bi prune -> dung model o GOC (final/best save_model). metric={best_metric}')
elif ckpts:
    subdir = f'checkpoint-{ckpts[-1]}'
    print(f'==> Fallback: checkpoint moi nhat {subdir}')
else:
    raise RuntimeError('Khong tim thay checkpoint nao tren repo!')

pat = (subdir + '/') if subdir else ''
snapshot_download(
    repo_id=HF_PRETRAIN_REPO, repo_type='model', local_dir='/content/pretrain_dl',
    allow_patterns=[pat+'config.json', pat+'generation_config.json', pat+'model.safetensors',
                    pat+'spiece.model', pat+'tokenizer*.json', pat+'special_tokens_map.json',
                    pat+'tokenizer_config.json'],
    ignore_patterns=(['checkpoint-*/**'] if subdir == '' else None),
    token=os.environ['HF_TOKEN'],
)
PRETRAIN_DIR = os.path.join('/content/pretrain_dl', subdir)
print('PRETRAIN_DIR =', PRETRAIN_DIR, '| checkpoint =', subdir or '(root/final)')
assert os.path.exists(os.path.join(PRETRAIN_DIR, 'model.safetensors')), 'Thieu model.safetensors!'

### Finetune (warm-start từ best pretrain checkpoint)

In [ ]:
import importlib
from training import finetune
importlib.reload(finetune)
print('>> Finetune se nap trong so tu:', PRETRAIN_DIR)
finetune.main(args_list=['configs/finetune.yaml', '--model_name_or_path', PRETRAIN_DIR])

### Upload model finetune lên HF

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
api.create_repo(repo_id=HF_FINETUNE_REPO, repo_type='model', exist_ok=True)
api.upload_folder(folder_path='/content/ViSceT5/output/finetune', repo_id=HF_FINETUNE_REPO,
                  repo_type='model', ignore_patterns=['optimizer.pt'])
print('Uploaded finetune ->', HF_FINETUNE_REPO)